In [ ]:
import os
import ast
import google.generativeai as genai

# 1. Initialize Gemini 1.5 Pro (The Payload Generator)
# Paste your actual Gemini API key here for the test
genai.configure(api_key="PASTE_YOUR_GEMINI_KEY_HERE")
model = genai.GenerativeModel('gemini-1.5-pro')

# 2. VAREK Physical Boundary (The Shield)
class VarekASTAnalyzer(ast.NodeVisitor):
    def __init__(self):
        self.violations = []
        # The ultimate blacklist for CrewAI's vulnerable fallback
        self.forbidden_modules = {'ctypes', 'os', 'subprocess', 'sys', 'socket'}

    def visit_Import(self, node):
        for alias in node.names:
            if alias.name.split('.')[0] in self.forbidden_modules:
                self.violations.append(f"Unauthorized System Module: {alias.name}")
        self.generic_visit(node)

    def visit_ImportFrom(self, node):
        if node.module and node.module.split('.')[0] in self.forbidden_modules:
            self.violations.append(f"Unauthorized System Module: {node.module}")
        self.generic_visit(node)

def varek_secure_execute(code_payload):
    print("--- [VAREK] COMPILING TO ABSTRACT SYNTAX TREE ---")
    try:
        tree = ast.parse(code_payload)
    except SyntaxError as e:
        raise RuntimeError(f"Syntax anomaly detected: {e}")

    print("--- [VAREK] EXECUTING DEEP STATIC ANALYSIS ---")
    analyzer = VarekASTAnalyzer()
    analyzer.visit(tree)

    if analyzer.violations:
        print("\n!!! [VAREK] KINETIC INTERCEPT TRIGGERED !!!")
        for violation in analyzer.violations:
            print(f"[FATAL] {violation}")
        raise RuntimeError("VAREK hard-halt: Malicious logic detected before execution.")

    print("--- [VAREK] CODE SAFE. ROUTING TO KERNEL... ---")
    exec(code_payload)